In [4]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, KFold
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from imblearn.under_sampling import RandomUnderSampler
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             balanced_accuracy_score, confusion_matrix,
                             matthews_corrcoef, cohen_kappa_score, log_loss,
                             mean_squared_error, mean_absolute_error, r2_score)
import time
import os
import skfuzzy as fuzz
from skfuzzy import control as ctrl
 
# Set the number of K folds as a global variable
K_FOLDS = 2
 
# Read the dataset from CSV file
df = pd.read_csv('../../../datasets/dataset-1.csv')
 
# Take 20% of the data
df = df.sample(frac=0.2, random_state=42)
 
# Rename the last column as 'label'
df.rename(columns={'GradeClass': 'label'}, inplace=True)
 
# Create a class for fuzzy logic classifier
class FuzzyLogicClassifier:
    def __init__(self):
        self.fuzzy_systems = {}
        self.rules = {}
        self.output_ranges = {}
        self.trained = False
        
    def _create_fuzzy_system(self, X, y):
        # Create universe variables
        feature_names = ["Age", "StudyTimeWeekly", "Absences", "ParentalSupport"]
        
        # Create fuzzy variables
        age = ctrl.Antecedent(np.linspace(14, 19, 100), 'age')
        study_time = ctrl.Antecedent(np.linspace(0, 20, 100), 'study_time')
        absences = ctrl.Antecedent(np.linspace(0, 30, 100), 'absences')
        parental_support = ctrl.Antecedent(np.linspace(0, 3, 100), 'parental_support')
        
        # Output variable
        grade = ctrl.Consequent(np.linspace(1, 4, 100), 'grade')
        
        # Auto-membership functions
        age.automf(3, names=['young', 'middle', 'old'])
        study_time.automf(3, names=['low', 'medium', 'high'])
        absences.automf(3, names=['few', 'average', 'many'])
        parental_support.automf(3, names=['low', 'medium', 'high'])
        
        # Grade membership functions
        grade['excellent'] = fuzz.trimf(grade.universe, [1, 1, 2])
        grade['good'] = fuzz.trimf(grade.universe, [1, 2, 3])
        grade['average'] = fuzz.trimf(grade.universe, [2, 3, 4])
        grade['poor'] = fuzz.trimf(grade.universe, [3, 4, 4])
        
        # Define fuzzy rules
        rule1 = ctrl.Rule(age['old'] & study_time['high'] & absences['few'] & parental_support['high'], grade['excellent'])
        rule2 = ctrl.Rule(age['middle'] & study_time['high'] & absences['few'], grade['excellent'])
        rule3 = ctrl.Rule(age['middle'] & study_time['medium'] & absences['few'], grade['good'])
        rule4 = ctrl.Rule(study_time['low'] & absences['many'], grade['poor'])
        rule5 = ctrl.Rule(study_time['medium'] & absences['average'], grade['average'])
        rule6 = ctrl.Rule(parental_support['low'] & absences['many'], grade['poor'])
        rule7 = ctrl.Rule(parental_support['high'] & study_time['high'], grade['excellent'])
        rule8 = ctrl.Rule(parental_support['medium'] & study_time['medium'], grade['good'])
        rule9 = ctrl.Rule(age['young'] & absences['many'], grade['poor'])
        
        # Create a control system
        grade_ctrl = ctrl.ControlSystem([rule1, rule2, rule3, rule4, rule5, rule6, rule7, rule8, rule9])
        
        return ctrl.ControlSystemSimulation(grade_ctrl)
        
    def fit(self, X, y):
        # Extract the relevant features
        if isinstance(X, pd.DataFrame):
            X_subset = X[["Age", "StudyTimeWeekly", "Absences", "ParentalSupport"]].values
        else:
            # Assuming columns are in the correct order
            X_subset = X[:, [1, 5, 6, 8]]  # Age, StudyTimeWeekly, Absences, ParentalSupport
            
        self.fuzzy_system = self._create_fuzzy_system(X_subset, y)
        self.trained = True
        
        # Save the range of output values for normalization
        self.output_min = 1
        self.output_max = 4
        
        return self
        
    def predict(self, X):
        if not self.trained:
            raise ValueError("The model has not been trained yet.")
            
        # Extract the relevant features
        if isinstance(X, pd.DataFrame):
            X_subset = X[["Age", "StudyTimeWeekly", "Absences", "ParentalSupport"]].values
        else:
            # Assuming columns are in the correct order
            X_subset = X[:, [1, 5, 6, 8]]  # Age, StudyTimeWeekly, Absences, ParentalSupport
            
        predictions = []
        for i in range(X_subset.shape[0]):
            self.fuzzy_system.input['age'] = X_subset[i, 0]
            self.fuzzy_system.input['study_time'] = X_subset[i, 1]
            self.fuzzy_system.input['absences'] = X_subset[i, 2]
            self.fuzzy_system.input['parental_support'] = X_subset[i, 3]
            
            try:
                self.fuzzy_system.compute()
                # Get the output
                grade_value = self.fuzzy_system.output['grade']
                
                # Round to the nearest class
                pred_class = round(grade_value)
                
                # Ensure the prediction is within the valid range
                pred_class = max(1, min(4, pred_class))
                
                predictions.append(pred_class)
            except:
                # If there's an error, default to the most common class
                predictions.append(2)  # Assuming 2 is the most common class
                
        return np.array(predictions)
 
X = df.drop(columns=['label']).values
y = df['label'].values
 
timing_results = []
 
# Define classifiers and metrics
classifiers = {
    'DecisionTree': DecisionTreeClassifier(),
    'KNN': KNeighborsClassifier(),
    'LogisticRegression': LogisticRegression(max_iter=1000),
    'FuzzyLogic': FuzzyLogicClassifier()
}

# Store results
results = []
 
kf = KFold(n_splits=K_FOLDS, shuffle=True, random_state=42)
 
# Create a directory to save confusion matrices
os.makedirs("confusion_matrices", exist_ok=True)
 
# Helper function for confusion matrix metrics
def confusion_matrix_metrics(cm, classes):
    metrics = {}
    for idx, class_label in enumerate(classes):
        TP = cm[idx, idx]  # True Positives for this class
        FP = cm[:, idx].sum() - TP  # False Positives for this class
        FN = cm[idx, :].sum() - TP  # False Negatives for this class
        TN = cm.sum() - (TP + FP + FN)  # True Negatives for this class
 
        metrics[class_label] = {
            'TPR': TP / (TP + FN + 1e-10) if (TP + FN) > 0 else 0,
            'TNR': TN / (TN + FP + 1e-10) if (TN + FP) > 0 else 0,
            'FPR': FP / (FP + TN + 1e-10) if (FP + TN) > 0 else 0,
            'FNR': FN / (FN + TP + 1e-10) if (FN + TP) > 0 else 0
        }
    return metrics
 
# Iterate over classifiers
for clf_name, clf in classifiers.items():
    fold_idx = 1
    for train_index, test_index in kf.split(X):
        # Split the data
        X_train, X_test = X[train_index], X[test_index]
        y_train, y_test = y[train_index], y[test_index]
 
        # Record start time
        start_train_time = time.time()
        clf.fit(X_train, y_train)
        train_time = time.time() - start_train_time
 
        start_test_time = time.time()
        y_pred = clf.predict(X_test)
        test_time = time.time() - start_test_time
 
        timing_results.append({
            'Classifier': clf_name,
            'Fold': fold_idx,
            'Training Time (s)': train_time,
            'Testing Time (s)': test_time,
            'Total Time (s)': train_time + test_time
        })
 
        # Compute metrics
        unique_classes = np.unique(np.concatenate((y_test, y_pred)))
        cm = confusion_matrix(y_test, y_pred, labels=unique_classes)
        cm_metrics = confusion_matrix_metrics(cm, unique_classes)
 
        class_metrics_list = []
 
        for class_label in unique_classes:
            class_mask = (y_test == class_label)
            class_pred_mask = (y_pred == class_label)
            
            # Binary classification metrics for this class
            y_true_binary = (y_test == class_label).astype(int)
            y_pred_binary = (y_pred == class_label).astype(int)
            
            if class_mask.sum() == 0:
                # Skip classes with no instances in the test set for this fold
                class_specific_metrics = {
                    'Classifier': clf_name,
                    'Fold': fold_idx,
                    'Class': class_label,
                    'Accuracy': np.nan,
                    'Precision': np.nan,
                    'Recall': np.nan,
                    'F1 Score': np.nan,
                    'Balanced Accuracy': np.nan,
                    'True Positive Rate (TPR)': np.nan,
                    'True Negative Rate (TNR)': np.nan,
                    'False Positive Rate (FPR)': np.nan,
                    'False Negative Rate (FNR)': np.nan,
                    'Training Time (s)': train_time,
                    'Testing Time (s)': test_time
                }
            else:
                class_specific_metrics = {
                    'Classifier': clf_name,
                    'Fold': fold_idx,
                    'Class': class_label,
                    'Accuracy': accuracy_score(y_true_binary, y_pred_binary),
                    'Precision': precision_score(y_true_binary, y_pred_binary, zero_division=0),
                    'Recall': recall_score(y_true_binary, y_pred_binary, zero_division=0),
                    'F1 Score': f1_score(y_true_binary, y_pred_binary, zero_division=0),
                    'Balanced Accuracy': balanced_accuracy_score(y_true_binary, y_pred_binary),
                    'True Positive Rate (TPR)': cm_metrics[class_label]['TPR'],
                    'True Negative Rate (TNR)': cm_metrics[class_label]['TNR'],
                    'False Positive Rate (FPR)': cm_metrics[class_label]['FPR'],
                    'False Negative Rate (FNR)': cm_metrics[class_label]['FNR'],
                    'Training Time (s)': train_time,
                    'Testing Time (s)': test_time
                }
 
            class_metrics_list.append(class_specific_metrics)
 
        # Plot and save confusion matrix
        plt.figure(figsize=(8, 6))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=unique_classes, yticklabels=unique_classes)
        plt.title(f"{clf_name} - Fold {fold_idx} Confusion Matrix")
        plt.xlabel("Predicted")
        plt.ylabel("True")
        plt.savefig(f"confusion_matrices/{clf_name}_fold_{fold_idx}.png")
        plt.close()
 
        # Append results for this fold
        results.extend(class_metrics_list)
        fold_idx += 1
 
timing_df = pd.DataFrame(timing_results)
timing_df.to_csv("time.csv", index=False)
 
# Create a DataFrame for results
results_df = pd.DataFrame(results)
print("Classification Metrics Across Folds:")
print(results_df)
 
# Save results to CSV
results_df.to_csv("metrics.csv", index=False)

# Additional Fuzzy Logic Visualization
def visualize_fuzzy_membership():
    # Create universe variables
    age = ctrl.Antecedent(np.linspace(14, 19, 100), 'age')
    study_time = ctrl.Antecedent(np.linspace(0, 20, 100), 'study_time')
    absences = ctrl.Antecedent(np.linspace(0, 30, 100), 'absences')
    parental_support = ctrl.Antecedent(np.linspace(0, 3, 100), 'parental_support')
    grade = ctrl.Consequent(np.linspace(1, 4, 100), 'grade')
    
    # Auto-membership functions
    age.automf(3, names=['young', 'middle', 'old'])
    study_time.automf(3, names=['low', 'medium', 'high'])
    absences.automf(3, names=['few', 'average', 'many'])
    parental_support.automf(3, names=['low', 'medium', 'high'])
    
    # Grade membership functions
    grade['excellent'] = fuzz.trimf(grade.universe, [1, 1, 2])
    grade['good'] = fuzz.trimf(grade.universe, [1, 2, 3])
    grade['average'] = fuzz.trimf(grade.universe, [2, 3, 4])
    grade['poor'] = fuzz.trimf(grade.universe, [3, 4, 4])
    
    # Create a directory to save fuzzy membership plots
    os.makedirs("fuzzy_membership", exist_ok=True)
    
    # Visualize membership functions
    plt.figure(figsize=(10, 4))
    age.view()
    plt.title("Age Membership Functions")
    plt.savefig("fuzzy_membership/age_membership.png")
    plt.close()
    
    plt.figure(figsize=(10, 4))
    study_time.view()
    plt.title("Study Time Membership Functions")
    plt.savefig("fuzzy_membership/study_time_membership.png")
    plt.close()
    
    plt.figure(figsize=(10, 4))
    absences.view()
    plt.title("Absences Membership Functions")
    plt.savefig("fuzzy_membership/absences_membership.png")
    plt.close()
    
    plt.figure(figsize=(10, 4))
    parental_support.view()
    plt.title("Parental Support Membership Functions")
    plt.savefig("fuzzy_membership/parental_support_membership.png")
    plt.close()
    
    plt.figure(figsize=(10, 4))
    grade.view()
    plt.title("Grade Membership Functions")
    plt.savefig("fuzzy_membership/grade_membership.png")
    plt.close()

# Call the function to generate fuzzy membership plots
visualize_fuzzy_membership()

# Generate a summary of fuzzy logic performance - FIXED THIS PART TO AVOID GROUPBY ERROR
print("\nFuzzy Logic Classifier Summary:")
fuzzy_results = results_df[results_df['Classifier'] == 'FuzzyLogic'].copy()

# Calculate mean values for each class manually instead of using groupby
class_summaries = {}
for class_label in fuzzy_results['Class'].unique():
    class_data = fuzzy_results[fuzzy_results['Class'] == class_label]
    class_summary = {
        'Accuracy': class_data['Accuracy'].mean(),
        'Precision': class_data['Precision'].mean(),
        'Recall': class_data['Recall'].mean(),
        'F1 Score': class_data['F1 Score'].mean(),
        'Balanced Accuracy': class_data['Balanced Accuracy'].mean(),
        'True Positive Rate (TPR)': class_data['True Positive Rate (TPR)'].mean(),
        'True Negative Rate (TNR)': class_data['True Negative Rate (TNR)'].mean(),
        'False Positive Rate (FPR)': class_data['False Positive Rate (FPR)'].mean(),
        'False Negative Rate (FNR)': class_data['False Negative Rate (FNR)'].mean(),
        'Training Time (s)': class_data['Training Time (s)'].mean(),
        'Testing Time (s)': class_data['Testing Time (s)'].mean()
    }
    class_summaries[class_label] = class_summary

# Convert to DataFrame for display and saving
fuzzy_summary = pd.DataFrame(class_summaries).T
print(fuzzy_summary)
fuzzy_summary.to_csv("fuzzy_summary.csv")

# Generate a comparison of classifiers
print("\nClassifier Performance Comparison:")
classifier_comparison = results_df.groupby('Classifier').agg({
    'Accuracy': 'mean',
    'Precision': 'mean',
    'Recall': 'mean',
    'F1 Score': 'mean',
    'Balanced Accuracy': 'mean',
    'Training Time (s)': 'mean',
    'Testing Time (s)': 'mean'
})
print(classifier_comparison)
classifier_comparison.to_csv("classifier_comparison.csv")

# Plot classifier performance comparison
plt.figure(figsize=(12, 8))
metrics_to_plot = ['Accuracy', 'Precision', 'Recall', 'F1 Score', 'Balanced Accuracy']
classifier_comparison[metrics_to_plot].plot(kind='bar')
plt.title('Classifier Performance Comparison')
plt.ylabel('Score')
plt.xlabel('Classifier')
plt.legend(loc='lower right')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.savefig('classifier_comparison.png')
plt.close()

# Plot training and testing times
plt.figure(figsize=(10, 6))
timing_comparison = timing_df.groupby('Classifier').agg({
    'Training Time (s)': 'mean',
    'Testing Time (s)': 'mean',
    'Total Time (s)': 'mean'
})
timing_comparison.plot(kind='bar')
plt.title('Classifier Timing Comparison')
plt.ylabel('Time (seconds)')
plt.xlabel('Classifier')
plt.legend()
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.savefig('timing_comparison.png')
plt.close()

/home/yashbuddhadev21/.local/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/home/yashbuddhadev21/.local/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html

Classification Metrics Across Folds:
            Classifier  Fold  Class  Accuracy  Precision    Recall  F1 Score  \
0         DecisionTree     1    0.0  0.974895   0.750000  0.750000  0.750000   
1         DecisionTree     1    1.0  0.953975   1.000000  0.541667  0.702703   
2         DecisionTree     1    2.0  0.945607   0.820000  0.911111  0.863158   
3         DecisionTree     1    3.0  0.953975   0.836735  0.931818  0.881720   
4         DecisionTree     1    4.0  0.970711   0.965217  0.973684  0.969432   
5         DecisionTree     2    0.0  0.974895   0.666667  0.800000  0.727273   
6         DecisionTree     2    1.0  0.953975   0.705882  0.960000  0.813559   
7         DecisionTree     2    2.0  0.953975   0.891892  0.825000  0.857143   
8         DecisionTree     2    3.0  0.979079   1.000000  0.880952  0.936709   
9         DecisionTree     2    4.0  0.979079   0.991597  0.967213  0.979253   
10                 KNN     1    0.0  0.933054   0.300000  0.250000  0.272727   
11 

/home/yashbuddhadev21/.local/lib/python3.10/site-packages/skfuzzy/control/fuzzyvariable.py:125: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.show()



Fuzzy Logic Classifier Summary:
     Accuracy  Precision    Recall  F1 Score  Balanced Accuracy  \
0.0  0.953975   0.000000  0.000000  0.000000           0.500000   
1.0  0.895397   0.166667  0.020833  0.037037           0.508091   
2.0  0.596234   0.267857  0.733333  0.392344           0.650185   
3.0  0.491632   0.172490  0.476732  0.252959           0.485912   
4.0  0.514644   0.500000  0.017544  0.033898           0.508772   

     True Positive Rate (TPR)  True Negative Rate (TNR)  \
0.0                  0.000000                  1.000000   
1.0                  0.020833                  0.995349   
2.0                  0.733333                  0.567036   
3.0                  0.476732                  0.495093   
4.0                  0.017544                  1.000000   

     False Positive Rate (FPR)  False Negative Rate (FNR)  Training Time (s)  \
0.0                   0.000000                   1.000000           0.012049   
1.0                   0.004651                   

<Figure size 1000x400 with 0 Axes>

...

<Figure size 1000x400 with 0 Axes>

<Figure size 1000x400 with 0 Axes>

<Figure size 1000x400 with 0 Axes>

<Figure size 1200x800 with 0 Axes>

<Figure size 1000x600 with 0 Axes>